In [1]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [2]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year
sst_df = sst_df.query('year >= 1993 and year <= 2024').reset_index().drop(['index'], axis=1)

In [3]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [4]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [5]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [6]:
chirps_eastern_east_africa = chirps.sel(latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
    10: 'OND', 11: 'OND', 12: 'OND' # October, November, December
}

season = ['MAM', 'OND']

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_eastern_east_africa_season = chirps_eastern_east_africa.dropna(subset=['season']).query(f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_eastern_east_africa_season = chirps_eastern_east_africa_season.query('year >= 1993 and year <= 2024').drop(['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_eastern_east_africa_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_eastern_east_africa_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_eastern_east_africa_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    season_an = chirps_eastern_east_africa_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn':sst_df_season_bn, 'n':sst_df_season_n, 'an':sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [7]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Eastern East Africa')
plt.savefig(f'figures/global_sst/sst_eea_season.png')
plt.close()

In [8]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Eastern East Africa')
plt.savefig('figures/global_sst/normalized_sst_eea_season.png')
plt.close()

In [31]:
chirps_southern_africa = chirps.sel(latitude=slice(-23, -15), longitude=slice(25, 34)).to_dataframe().reset_index()
chirps_southern_africa['month'] = chirps_southern_africa['time'].dt.month
chirps_southern_africa['year'] = chirps_southern_africa['time'].dt.year

month_to_season = {
    2: 'FMA', 3: 'FMA', 4: 'FMA',
    12: 'DFJ', 1: 'DJF', 2: 'DJF'
}

season = ['FMA', 'DJF']

chirps_southern_africa['season'] = chirps_southern_africa['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_southern_africa_season = chirps_southern_africa.dropna(subset=['season']).query(f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_southern_africa_season = chirps_southern_africa_season.query('year >= 1993 and year <= 2024').drop(['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_southern_africa_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_southern_africa_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_southern_africa_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    season_an = chirps_southern_africa_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn':sst_df_season_bn, 'n':sst_df_season_n, 'an':sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [32]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='month',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Southern Africa')
plt.savefig(f'figures/global_sst/sst_sa.png')
plt.close()

In [33]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='month',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Southern Africa')
plt.savefig('figures/global_sst/normalized_sst_sa.png')
plt.close()

In [9]:
chirps_west_africa = chirps.sel(latitude=slice(10, 13.5), longitude=slice(-10, 0)).to_dataframe().reset_index()
chirps_west_africa['month'] = chirps_west_africa['time'].dt.month
chirps_west_africa['year'] = chirps_west_africa['time'].dt.year

month_to_season = {
    7: 'JAS', 8: 'JAS', 9: 'JAS',  # March, April, May
}

season = ['JAS']

chirps_west_africa['season'] = chirps_west_africa['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_west_africa_season = chirps_west_africa.dropna(subset=['season']).query(
        f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_west_africa_season = chirps_west_africa_season.query('year >= 1993 and year <= 2024').drop(
        ['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_west_africa_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_west_africa_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_west_africa_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')[
        'year'].to_list()
    season_an = chirps_west_africa_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn': sst_df_season_bn, 'n': sst_df_season_n, 'an': sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(),
                              names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [10]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('West Africa')
plt.savefig('figures/global_sst/normalized_sst_wa_season.png')
plt.close()

In [11]:
chirps_sri_lanka = chirps.sel(latitude=slice(5.5, 10), longitude=slice(79.5, 82)).to_dataframe().reset_index()
chirps_sri_lanka['month'] = chirps_sri_lanka['time'].dt.month
chirps_sri_lanka['year'] = chirps_sri_lanka['time'].dt.year

month_to_season = {
    10: 'OND', 11: 'OND', 12: 'OND' # October, November, December
}

season = ['OND']

chirps_sri_lanka['season'] = chirps_sri_lanka['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_sri_lanka_season = chirps_sri_lanka.dropna(subset=['season']).query(f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_sri_lanka_season = chirps_sri_lanka_season.query('year >= 1993 and year <= 2024').drop(['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_sri_lanka_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_sri_lanka_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_sri_lanka_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    season_an = chirps_sri_lanka_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn':sst_df_season_bn, 'n':sst_df_season_n, 'an':sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [12]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Sri Lanka')
plt.savefig('figures/global_sst/normalized_sst_sl_season.png')
plt.close()

In [13]:
chirps_lake_victoria = chirps.sel(latitude=slice(-4, 1.5), longitude=slice(29, 36)).to_dataframe().reset_index()
chirps_lake_victoria['month'] = chirps_lake_victoria['time'].dt.month
chirps_lake_victoria['year'] = chirps_lake_victoria['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',
    12: 'DJF', 1: 'DJF', 2: 'DJF',
    9: 'SON', 10: 'SON', 11: 'SON'
}

season = ['DJF', 'MAM', 'SON']

chirps_lake_victoria['season'] = chirps_lake_victoria['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_lake_victoria_season = chirps_lake_victoria.dropna(subset=['season']).query(
        f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_lake_victoria_season = chirps_lake_victoria_season.query('year >= 1993 and year <= 2024').drop(
        ['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_lake_victoria_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_lake_victoria_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_lake_victoria_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')[
        'year'].to_list()
    season_an = chirps_lake_victoria_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn': sst_df_season_bn, 'n': sst_df_season_n, 'an': sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(),
                              names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [14]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Lake Victoria')
plt.savefig('figures/global_sst/normalized_sst_lv_season.png')
plt.close()

In [15]:
chirps_eastern_ukraine = chirps.sel(latitude=slice(45, 51), longitude=slice(31, 40)).to_dataframe().reset_index()
chirps_eastern_ukraine['month'] = chirps_eastern_ukraine['time'].dt.month
chirps_eastern_ukraine['year'] = chirps_eastern_ukraine['time'].dt.year

month_to_season = {
    12: 'DJF', 1: 'DJF', 2: 'DJF',
    4: 'AMJ', 5: 'AMJ', 6: 'AMJ',
    7: 'JA', 8: 'JA'
}

season = ['DJF', 'AMJ', 'JA']

chirps_eastern_ukraine['season'] = chirps_eastern_ukraine['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_eastern_ukraine_season = chirps_eastern_ukraine.dropna(subset=['season']).query(
        f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_eastern_ukraine_season = chirps_eastern_ukraine_season.query('year >= 1993 and year <= 2024').drop(
        ['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_eastern_ukraine_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_eastern_ukraine_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_eastern_ukraine_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')[
        'year'].to_list()
    season_an = chirps_eastern_ukraine_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn': sst_df_season_bn, 'n': sst_df_season_n, 'an': sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(),
                              names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [16]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Eastern Ukraine')
plt.savefig('figures/global_sst/normalized_sst_eu_season.png')
plt.close()

In [ ]:
chirps_south_sudan = chirps.sel(latitude=slice(3.5, 12.5), longitude=slice(25, 35)).to_dataframe().reset_index()
chirps_south_sudan['month'] = chirps_south_sudan['time'].dt.month
chirps_south_sudan['year'] = chirps_south_sudan['time'].dt.year

month_to_season = {
    5: 'MJJ', 6: 'MJJ', 7: 'MJJ',
    7: 'JAS', 8: 'JAS', 9: 'JAS',
    8: 'ASO', 9: 'ASO', 10: 'ASO'
}

season = ['MJJ', 'JAS', 'ASO']

chirps_south_sudan['season'] = chirps_south_sudan['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_south_sudan_season = chirps_south_sudan.dropna(subset=['season']).query(
        f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_south_sudan_season = chirps_south_sudan_season.query('year >= 1993 and year <= 2024').drop(
        ['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_south_sudan_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_south_sudan_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_south_sudan_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')[
        'year'].to_list()
    season_an = chirps_south_sudan_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn': sst_df_season_bn, 'n': sst_df_season_n, 'an': sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(),
                              names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [ ]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('South Sudan')
plt.savefig('figures/global_sst/normalized_sst_ss_season.png')
plt.close()